# Phase 5: Evaluation Metrics

In this notebook, we evaluate all three recommendation approaches (Item-CF, SVD, and Hybrid) using an 80/20 train-test split. We calculate standard accuracy metrics (RMSE, MAE) and the rigorous ranking metric Mean Average Precision at 10 (MAP@10) to determine the best-performing model.


In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')


## 1. Load Data and Train-Test Split
We load the 6.9 million rating subset to ensure practical execution times. We split 80% for training and 20% for testing.

In [2]:
# Load the 6.9M rating subset
df = pd.read_csv('../data/processed/netflix_svd_subset.csv')

# Parse for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'movie_id', 'rating']], reader)

# Create 80/20 train/test split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Training set: {trainset.n_ratings:,} ratings")
print(f"Test set: {len(testset):,} ratings")


Training set: 5,539,228 ratings
Test set: 1,384,807 ratings


## 2. Train SVD and Item-CF Models
We use the `scikit-surprise` implementations for standardized, fair evaluation across models.

In [3]:
# Train SVD
print("Training SVD...")
algo_svd = SVD(n_factors=50, n_epochs=20, random_state=42)
algo_svd.fit(trainset)

# Train Item-CF (KNNBasic)
print("Training Item-CF...")
sim_options = {'name': 'cosine', 'user_based': False}
algo_knn = KNNBasic(sim_options=sim_options, verbose=False)
algo_knn.fit(trainset)

print("Models trained successfully!")


Training SVD...


Training Item-CF...


Models trained successfully!


## 3. Generate Predictions
We predict ratings for the 1.4 million test set items. The Hybrid predictions are calculated as a weighted average (0.7 SVD + 0.3 Item-CF).

In [4]:
print("Predicting SVD...")
preds_svd = algo_svd.test(testset)

print("Predicting Item-CF...")
preds_knn = algo_knn.test(testset)

print("Calculating Hybrid predictions...")
preds_hybrid = []
for svd_pred, knn_pred in zip(preds_svd, preds_knn):
    # Hybrid formula: 0.7 * SVD + 0.3 * Item-CF
    est_hybrid = (0.7 * svd_pred.est) + (0.3 * knn_pred.est)
    preds_hybrid.append((svd_pred.uid, svd_pred.iid, svd_pred.r_ui, est_hybrid))


Predicting SVD...


Predicting Item-CF...


Calculating Hybrid predictions...


## 4. Evaluate RMSE and MAE
We calculate Root Mean Squared Error (RMSE) and Mean Absolute Error (MAE) directly comparing the predicted ratings against the true test ratings.

In [5]:
def calculate_errors(predictions):
    """Calculates RMSE and MAE from a list of (uid, iid, true_r, est) tuples."""
    mse = np.mean([(true_r - est)**2 for (_, _, true_r, est) in predictions])
    mae = np.mean([abs(true_r - est) for (_, _, true_r, est) in predictions])
    return np.sqrt(mse), mae

# We extract standard tuples from Surprise prediction objects
svd_tuples = [(p.uid, p.iid, p.r_ui, p.est) for p in preds_svd]
knn_tuples = [(p.uid, p.iid, p.r_ui, p.est) for p in preds_knn]

rmse_svd, mae_svd = calculate_errors(svd_tuples)
rmse_knn, mae_knn = calculate_errors(knn_tuples)
rmse_hybrid, mae_hybrid = calculate_errors(preds_hybrid)

print(f"SVD    - RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f}")
print(f"ItemCF - RMSE: {rmse_knn:.4f}, MAE: {mae_knn:.4f}")
print(f"Hybrid - RMSE: {rmse_hybrid:.4f}, MAE: {mae_hybrid:.4f}")


SVD    - RMSE: 0.8013, MAE: 0.6212
ItemCF - RMSE: 0.9421, MAE: 0.7283
Hybrid - RMSE: 0.8136, MAE: 0.6310


## 5. Evaluate MAP@10
Mean Average Precision at 10 (MAP@10) rigorously evaluates the ranking quality of our Top-10 recommendations. **Methodology:** We generated predictions using an 80/20 train-test split. For each user, we take the Top-10 predicted recommendations and calculate MAP@10. We explicitly define 'relevance' as a true rating >= 3.5. To save computation time, we sample 2,000 users.

In [6]:
def average_precision_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est in predictions:
        user_est_true[uid].append((est, true_r))
        
    ap_scores = dict()
    
    for uid, user_ratings in user_est_true.items():
        # Sort user ratings by estimated value descending
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        
        # Total relevant items for user in the entire test set
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        
        if n_rel == 0:
            continue # Skip users with no relevant items
            
        hits = 0
        sum_precisions = 0
        
        # Check Top K
        for i, (est, true_r) in enumerate(user_ratings[:k]):
            if true_r >= threshold:
                hits += 1
                sum_precisions += hits / (i + 1.0)
                
        ap_scores[uid] = sum_precisions / min(n_rel, k)
        
    return np.mean(list(ap_scores.values())) if ap_scores else 0

# Sample 2000 users from testset
np.random.seed(42)
sample_uids = set(np.random.choice(list(set([p[0] for p in svd_tuples])), size=2000, replace=False))
sample_svd = [p for p in svd_tuples if p[0] in sample_uids]
sample_knn = [p for p in knn_tuples if p[0] in sample_uids]
sample_hybrid = [p for p in preds_hybrid if p[0] in sample_uids]

map10_svd = average_precision_at_k(sample_svd, k=10)
map10_knn = average_precision_at_k(sample_knn, k=10)
map10_hybrid = average_precision_at_k(sample_hybrid, k=10)

print(f"SVD    - MAP@10: {map10_svd:.4f}")
print(f"ItemCF - MAP@10: {map10_knn:.4f}")
print(f"Hybrid - MAP@10: {map10_hybrid:.4f}")


SVD    - MAP@10: 0.8754
ItemCF - MAP@10: 0.7697
Hybrid - MAP@10: 0.8800


## 6. Comparison Table

In [7]:
results = pd.DataFrame({
    'Model': ['Item-CF', 'SVD', 'Hybrid'],
    'RMSE': [rmse_knn, rmse_svd, rmse_hybrid],
    'MAE': [mae_knn, mae_svd, mae_hybrid],
    'MAP@10': [map10_knn, map10_svd, map10_hybrid]
})

display(results.round(4))


,Model,RMSE,MAE,MAP@10
0,Item-CF,0.9421,0.7283,0.7697
1,SVD,0.8013,0.6212,0.8754
2,Hybrid,0.8136,0.6310,0.8800


## 7. Discussion

**Which model performed best:**
- **SVD is best for rating prediction accuracy.** It achieved the lowest RMSE (0.8013) and MAE (0.6212).
- **Hybrid is best for recommendation ranking quality.** It achieved the highest MAP@10 (0.8800), outperforming SVD (0.8754) and Item-CF (0.7697).
- **Item-CF provides explainability.** While it has the worst accuracy metrics (RMSE 0.9421), it is the only pure model that offers direct transparency into *why* a movie is recommended.

**Why the Hybrid model is the ideal compromise:**
- By injecting 30% Item-CF logic into SVD, the Hybrid model suffers a minor penalty in pure rating prediction (RMSE drops from 0.8013 to 0.8136).
- However, this 30% injection actually *improves* the final ranking quality (MAP@10 increases to 0.8800).
- This means the Hybrid model gives us the best Top-10 lists, while simultaneously allowing us to provide transparent explanations to the user.

**Conclusion:**
For an undergraduate-level project aiming to build a practical, user-facing recommendation system, the Hybrid model is the clear winner.
